# V2-1 (r3)  CA-ANN-Markov  - CPU runtime is enough (no GPU used)
Resumes automatically: seeds whose outputs already exist are skipped.

In [ ]:
!pip install -q rasterio scikit-learn tqdm seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- RUN CONFIGURATION: edit ONLY this cell. RESTART THE SESSION between different runs. ---
# Run 1 (main):            FEATURE_MODE='anomaly'; RUN_TAG='';            DYN_USE=None;                                SEEDS=[0,1,2,3,4]
# Run 2 (ablation A):      FEATURE_MODE='global';  RUN_TAG='';            DYN_USE=None;                                SEEDS=[0,1,2]
# Run 3 (ablation B):      FEATURE_MODE='anomaly'; RUN_TAG='_noclim';     DYN_USE=['NDVI','MNDWI','NDMI','Dist_Water']; SEEDS=[0,1,2]
# Run 4 (ablation C):      FEATURE_MODE='anomaly'; RUN_TAG='_stateonly';  DYN_USE=[];                                  SEEDS=[0,1,2]
FEATURE_MODE = 'anomaly'
RUN_TAG = '_stateonly'
DYN_USE = []
SEEDS = [0, 1, 2]


In [ ]:
# =====================================================================
# CORE v2 - shared setup (identical first code cell in every v2 notebook)
#   * one-step design: state(t) + drivers(t) -> state(t+10);   T1 = 2003->2013,  T2 = 2013->2023
#   * FEATURE_MODE 'anomaly' : each continuous layer standardised WITHIN its own year (removes year/sensor/weather offsets)
#     FEATURE_MODE 'global'  : old global min-max across years (kept only for the ablation)
# =====================================================================
import os, math, random, warnings, copy
import numpy as np, pandas as pd, cv2, rasterio
import matplotlib.pyplot as plt, seaborn as sns
from rasterio.warp import reproject, Resampling
from sklearn.metrics import confusion_matrix, cohen_kappa_score, precision_recall_fscore_support
warnings.filterwarnings('ignore')
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception:
    pass

LIGHT        = globals().get('LIGHT', False)               # True = skip feature building (used by the comparison notebook)
FEATURE_MODE = globals().get('FEATURE_MODE', 'anomaly')    # 'anomaly' | 'global'

# ---------------- CONFIG ----------------
PATH   = '/content/drive/MyDrive/Obj2_LULC_GWP/'
RUN_TAG = globals().get('RUN_TAG', '')                     # e.g. '_noclim' / '_stateonly' for ablation runs (separate output folder)
OUT    = os.path.join(PATH, f'revised_outputs_v2_{FEATURE_MODE}{RUN_TAG}'); os.makedirs(OUT, exist_ok=True)
YEARS  = ['2003', '2013', '2023']
CLASS_VALUES = [1, 2, 3, 4, 5]
CLASS_NAMES  = ['Water', 'Forest', 'Agriculture', 'Settlement', 'Fallow']
K = len(CLASS_VALUES); WATER_IDX, FOREST_IDX, FALLOW_IDX = 0, 1, 4
DYN_CONT = ['NDVI', 'MNDWI', 'LST', 'Rainfall', 'Dist_Water', 'NDMI']
_du = globals().get('DYN_USE', None)                       # None = all continuous layers; [] = state + static only
DYN_USE  = DYN_CONT if _du is None else _du
STATIC   = ['SRTM_DEM', 'Slope', 'DD', 'Ld', 'TWI']

PATCH, HIDDEN, EPOCHS, BATCH, LR, PATIENCE = 5, 64, 50, 1024, 5e-4, 6
SAMPLE_FRAC = 0.10
SEEDS = globals().get('SEEDS', [0, 1, 2])
BLOCK, SPLIT_SEED, TEST_FRAC, VAL_FRAC = 64, 42, 0.15, 0.15
CA_GAMMA = 1.0
STEEP_DEG = 10.0          # a-priori physical prior (NOT data-derived): water on slopes above this is implausible
PHYS_RAMP_DEG = 10.0      # penalty ramps 0 -> 1 between STEEP_DEG and STEEP_DEG + PHYS_RAMP_DEG

def set_seed(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError: pass

# ---------------- RASTERS (reprojected onto the LULC_2023 master grid) ----------------
with rasterio.open(os.path.join(PATH, 'LULC_2023.tif')) as src:
    master_shape, master_meta, master_tf, master_crs = src.read(1).shape, src.meta.copy(), src.transform, src.crs
    PIXEL_KM2 = abs(src.res[0] * src.res[1]) / 1e6
H, W = master_shape
print(f'FEATURE_MODE = {FEATURE_MODE} | grid {H} x {W} | pixel = {PIXEL_KM2*1e6:.0f} m2 | outputs -> {OUT}')

def read_raster(name, categorical=False):
    with rasterio.open(os.path.join(PATH, name)) as src:
        a = src.read(1).astype('float32'); nd = src.nodata
        if nd is not None: a[a == nd] = np.nan
        a[(a < -1e10) | (a > 1e10)] = np.nan
        if src.transform != master_tf or a.shape != master_shape or src.crs != master_crs:
            dst = np.full(master_shape, np.nan, 'float32')
            reproject(source=a, destination=dst, src_transform=src.transform, src_crs=src.crs, src_nodata=np.nan,
                      dst_transform=master_tf, dst_crs=master_crs, dst_nodata=np.nan,
                      resampling=Resampling.nearest if categorical else Resampling.bilinear)
            print(f'  {name}: reprojected {a.shape} -> {master_shape}'); a = dst
    a[~np.isfinite(a)] = np.nan
    return a

def lulc_to_index(a):
    out = np.full(a.shape, -1, np.int8)
    for i, v in enumerate(CLASS_VALUES): out[a == v] = i
    return out

LULC  = {y: lulc_to_index(read_raster(f'LULC_{y}.tif', True)) for y in YEARS}
VALID = np.all([LULC[y] >= 0 for y in YEARS], axis=0)
VR, VC = np.where(VALID); NV = len(VR)
print(f'Valid pixels: {NV:,} ({NV*PIXEL_KM2:.1f} km2)')
slope_raw = read_raster('Slope.tif')
slope_deg = slope_raw.copy()
if np.nanmax(slope_deg) > 90: slope_deg = np.degrees(np.arctan(slope_deg / 100.0)); print('Slope treated as percent-rise -> degrees')
slope_deg = np.nan_to_num(slope_deg, nan=0.0)

# ---------------- FEATURES ----------------
def onehot(idx):
    oh = np.zeros((K,) + idx.shape, np.float32)
    for k in range(K): oh[k] = (idx == k)
    return oh

if not LIGHT:
    dyn_raw = {y: np.stack([read_raster(f'{n}_{y}.tif') for n in DYN_CONT]) for y in YEARS}
    stat_raw = np.stack([slope_raw if n == 'Slope' else read_raster(f'{n}.tif') for n in STATIC])
    for _y in YEARS:
        for _i, _n in enumerate(DYN_CONT):
            _p = np.isnan(dyn_raw[_y][_i][VALID]).mean() * 100
            if _p > 0.5: print(f'  CHECK {_n}_{_y}: {_p:.1f}% NaN inside study area')
    for _i, _n in enumerate(STATIC):
        _p = np.isnan(stat_raw[_i][VALID]).mean() * 100
        if _p > 0.5: print(f'  CHECK {_n}: {_p:.1f}% NaN inside study area')

    g_lo = np.array([np.nanmin([np.nanmin(dyn_raw[y][i][VALID]) for y in YEARS]) for i in range(len(DYN_CONT))])
    g_hi = np.array([np.nanmax([np.nanmax(dyn_raw[y][i][VALID]) for y in YEARS]) for i in range(len(DYN_CONT))])

    def feat_layer(a, i, name):
        if FEATURE_MODE == 'global':
            z = np.clip((a - g_lo[i]) / (g_hi[i] - g_lo[i] + 1e-6), 0, 1)
            z = np.where(np.isnan(z), np.nanmedian(z[VALID]), z)
        else:                                            # within-year standardisation (spatial anomaly), clipped to +-4 SD, scaled to [-1, 1]
            x = np.log1p(np.clip(a, 0, None)) if name == 'Dist_Water' else a
            v = x[VALID]; v = v[np.isfinite(v)]
            z = np.clip((x - v.mean()) / (v.std() + 1e-9), -4, 4) / 4
            z = np.where(np.isfinite(z), z, 0)
        return np.where(VALID, z, 0).astype(np.float32)

    def build_dyn(y):
        ch = [feat_layer(dyn_raw[y][i], i, n) for i, n in enumerate(DYN_CONT) if n in DYN_USE]
        return np.concatenate([onehot(LULC[y])] + ([np.stack(ch)] if ch else []))
    DYN = {y: build_dyn(y) for y in YEARS}                                     # (5 + n_cont, H, W) per year

    s_lo = np.nanmin(np.where(VALID, stat_raw, np.nan), axis=(1, 2))[:, None, None]
    s_hi = np.nanmax(np.where(VALID, stat_raw, np.nan), axis=(1, 2))[:, None, None]
    STAT = np.clip((stat_raw - s_lo) / (s_hi - s_lo + 1e-6), 0, 1)
    STAT = np.where(np.isnan(STAT), np.nanmedian(np.where(VALID, STAT, np.nan), axis=(1, 2), keepdims=True), STAT)
    STAT = np.where(VALID, STAT, 0).astype(np.float32)
    C_DYN, C_STAT = DYN['2003'].shape[0], STAT.shape[0]
    print(f'Channels: dynamic {C_DYN} (5 one-hot LULC + {C_DYN-K} continuous), static {C_STAT}')

    # year-shift check on pixels whose class never changed (should be small after standardisation)
    _st = (LULC['2003'] == LULC['2013']) & (LULC['2013'] == LULC['2023']) & VALID
    _names = [n for n in DYN_CONT if n in DYN_USE]; _mx = {}
    for j, n in enumerate(_names):
        _d = []
        for k in range(K):
            m = _st & (LULC['2023'] == k)
            if m.sum() > 500:
                _d.append([abs(DYN['2013'][K+j][m].mean() - DYN['2003'][K+j][m].mean()), abs(DYN['2023'][K+j][m].mean() - DYN['2013'][K+j][m].mean())])
        _mx[n] = np.max(_d, axis=0) * (4 if FEATURE_MODE == 'anomaly' else 1)
    print('Max class-conditional year shift on never-changed pixels (T1, T2) [SD units if anomaly mode, 0-1 units if global]:')
    for n, v in _mx.items(): print(f'   {n:11s} T1 {v[0]:.2f}   T2 {v[1]:.2f}')

# ---------------- SPATIAL BLOCK SPLIT ----------------
def make_block_split(valid, block, test_frac, val_frac, seed):
    nbr, nbc = math.ceil(H / block), math.ceil(W / block)
    has = np.array([[valid[i*block:(i+1)*block, j*block:(j+1)*block].any() for j in range(nbc)] for i in range(nbr)])
    ids = np.argwhere(has); rng = np.random.RandomState(seed); rng.shuffle(ids)
    n_te, n_va = int(round(test_frac * len(ids))), int(round(val_frac * len(ids)))
    lab = np.full((nbr, nbc), -1, np.int8)
    for n, (i, j) in enumerate(ids): lab[i, j] = 2 if n < n_te else (1 if n < n_te + n_va else 0)
    full = np.where(valid, np.repeat(np.repeat(lab, block, 0), block, 1)[:H, :W], -1).astype(np.int8)
    return full, np.repeat(np.repeat(np.arange(nbr*nbc).reshape(nbr, nbc), block, 0), block, 1)[:H, :W]
SPLIT, BLOCK_ID = make_block_split(VALID, BLOCK, TEST_FRAC, VAL_FRAC, SPLIT_SEED)
for n, nm in enumerate(['train', 'val', 'test']): print(f'  {nm:5s}: {(SPLIT==n).sum():,} px')

# ---------------- PATCH SOURCE (single epoch) ----------------
class PatchSource:
    def __init__(self, dyn, ps=PATCH):
        pad = ps // 2
        self.d = np.pad(dyn, ((0, 0), (pad, pad), (pad, pad)), mode='reflect'); self.off = np.arange(ps)
    def get(self, rows, cols):
        rr = (rows[:, None] + self.off[None, :])[:, :, None]; cc = (cols[:, None] + self.off[None, :])[:, None, :]
        x = self.d[:, rr, cc]                                           # (C,B,ps,ps)
        return (np.ascontiguousarray(np.transpose(x, (1, 0, 2, 3))),
                np.ascontiguousarray(STAT[:, rows, cols].T), slope_deg[rows, cols].astype(np.float32))

def sample_coords(split_id, frac, seed):
    r, c = np.where(SPLIT == split_id)
    sel = np.random.RandomState(seed).choice(len(r), int(len(r) * frac), replace=False)
    return r[sel], c[sel]

# ---------------- MARKOV DEMAND + CA ALLOCATION ----------------
def markov_matrix(a, b, mask):
    m = np.bincount((a[mask].astype(int) * K + b[mask]).astype(int), minlength=K*K).reshape(K, K).astype(float)
    return m / m.sum(1, keepdims=True).clip(1)

def allocate(T, demand, iters=300, tol=0.001):
    logT = np.log(T + 1e-9); b = np.zeros(K); N = T.shape[1]; lr = 1.0
    for _ in range(iters):
        lab = np.argmax(logT + b[:, None], 0); cnt = np.bincount(lab, minlength=K)
        dv = np.abs(cnt - demand).max() / N
        if dv < tol: break
        b += lr * np.log((demand + 1) / (cnt + 1)); lr = max(0.05, lr * 0.99)
    return lab.astype(np.int8), dv

def ca_allocate(prob, prev_map, trans, gamma=CA_GAMMA):
    """prob: (K, NV) class probabilities at valid pixels. Markov demand = class counts(prev) @ trans; 5x5 contiguity factor; competitive allocation."""
    S = prob.astype(np.float32).copy()
    for k in range(K): S[k] *= (1.0 + gamma * cv2.blur((prev_map == k).astype(np.float32), (5, 5))[VR, VC])
    demand = np.rint(np.bincount(prev_map[VALID], minlength=K) @ trans).astype(int)
    demand[np.argmax(demand)] += NV - demand.sum()
    lab, dv = allocate(S, demand)
    print(f'    demand km2 {(demand*PIXEL_KM2).round(1).tolist()} | max alloc-demand gap {dv*100:.3f}% px')
    return lab

def scatter(lab):
    out = np.full((H, W), -1, np.int8); out[VR, VC] = lab; return out

# ---------------- METRICS / IO ----------------
def evaluate_map(true, pred, prev, mask):
    t, p, q = true[mask].astype(np.int64), pred[mask].astype(np.int64), prev[mask].astype(np.int64)
    out = {'n': int(mask.sum()), 'OA': float((t == p).mean()), 'Kappa': float(cohen_kappa_score(t, p))}
    pr, rc, f1, _ = precision_recall_fscore_support(t, p, labels=range(K), zero_division=0)
    for k, nm in enumerate(CLASS_NAMES): out[f'F1_{nm}'] = float(f1[k]); out[f'R_{nm}'] = float(rc[k])
    out['MacroF1'] = float(f1.mean())
    ch = t != q
    hits, miss = int((ch & (p == t)).sum()), int((ch & (p == q)).sum())
    wrong, fa = int((ch & (p != q) & (p != t)).sum()), int((~ch & (p != q)).sum())
    out.update(change_observed=float(ch.mean()), change_predicted=float((p != q).mean()), hits=hits, misses=miss,
               wrong_hits=wrong, false_alarms=fa, FoM=hits / max(1, hits + miss + wrong + fa))
    qd = np.abs(np.bincount(t, minlength=K) - np.bincount(p, minlength=K)).sum() / 2 / len(t)
    out.update(QuantityDisagreement=float(qd), AllocationDisagreement=float((1 - out['OA']) - qd))
    steep = slope_deg[mask] > STEEP_DEG
    out.update(water_steep_pred=int(((p == WATER_IDX) & steep).sum()), water_steep_true=int(((t == WATER_IDX) & steep).sum()),
               new_water_pred=int(((p == WATER_IDX) & (q != WATER_IDX)).sum()), new_water_true=int(((t == WATER_IDX) & (q != WATER_IDX)).sum()),
               new_water_steep_pred=int(((p == WATER_IDX) & (q != WATER_IDX) & steep).sum()),
               F2F_pred_px=int(((q == FALLOW_IDX) & (p == FOREST_IDX)).sum()), F2F_true_px=int(((q == FALLOW_IDX) & (t == FOREST_IDX)).sum()),
               F2F_hit_px=int(((q == FALLOW_IDX) & (p == FOREST_IDX) & (t == FOREST_IDX)).sum()))
    return out

def save_map(idx_map, fname):
    m = master_meta.copy(); m.update(dtype='uint8', count=1, nodata=0, compress='lzw')
    codes = np.zeros(idx_map.shape, np.uint8)
    for k, v in enumerate(CLASS_VALUES): codes[(idx_map == k) & VALID] = v
    with rasterio.open(os.path.join(OUT, fname), 'w', **m) as dst: dst.write(codes, 1)

def read_idx(folder, fname):
    with rasterio.open(os.path.join(folder, fname)) as s: a = s.read(1)
    o = np.full(a.shape, -1, np.int8)
    for k, v in enumerate(CLASS_VALUES): o[a == v] = k
    return o
print('Core v2 ready.')


In [ ]:
# =====================================================================
# CA-ANN-Markov (v2)  -  ANN suitability + Markov demand + 5x5 CA contiguity + competitive allocation
#   Stage V (validation, OUT-OF-TIME): calibrate on T1 (2003->2013), predict 2023 from 2013, compare with observed 2023.
#   Stage F (forecast)               : calibrate on T2 (2013->2023), predict 2033 from 2023, 3 Markov-rate scenarios.
# =====================================================================
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss
MODEL_TAG = 'CA_ANN_Markov'
TEST = SPLIT == 2
P_T1_train = markov_matrix(LULC['2003'], LULC['2013'], SPLIT == 0)      # stage V: rates from TRAIN blocks of T1 only
P_T1_all   = markov_matrix(LULC['2003'], LULC['2013'], VALID)
P_T2_all   = markov_matrix(LULC['2013'], LULC['2023'], VALID)
SCEN = {'T2': P_T2_all, 'T1': P_T1_all, 'Mean': (P_T1_all + P_T2_all) / 2}
for nm, P in [('T1 (2003->2013)', P_T1_all), ('T2 (2013->2023)', P_T2_all)]:
    print(f'Markov matrix {nm}:'); print(pd.DataFrame(P, CLASS_NAMES, CLASS_NAMES).round(4))

def feats(y): return np.concatenate([DYN[y], STAT])

def fit_ann(year_in, year_out, seed):
    F = feats(year_in)
    tr_r, tr_c = sample_coords(0, SAMPLE_FRAC, 1); va_r, va_c = sample_coords(1, SAMPLE_FRAC, 2)
    Xtr, ytr = F[:, tr_r, tr_c].T, LULC[year_out][tr_r, tr_c].astype(int)
    Xva, yva = F[:, va_r, va_c].T, LULC[year_out][va_r, va_c].astype(int)
    set_seed(seed); rng = np.random.RandomState(seed)
    ann = MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-4, learning_rate_init=LR, batch_size=BATCH, random_state=seed)
    best, bad, best_ann = 1e9, 0, None
    for ep in range(EPOCHS):
        perm = rng.permutation(len(ytr))
        for i in range(0, len(perm), BATCH):
            idx = perm[i:i+BATCH]; ann.partial_fit(Xtr[idx], ytr[idx], classes=np.arange(K))
        vl = log_loss(yva, ann.predict_proba(Xva), labels=np.arange(K))
        if vl < best - 1e-5: best, bad, best_ann = vl, 0, copy.deepcopy(ann)
        else:
            bad += 1
            if bad >= PATIENCE: break
    print(f'  ANN {year_in}->{year_out} seed {seed}: stopped at epoch {ep+1}, best val log-loss {best:.4f}')
    return best_ann

def proba_valid(ann, F):
    P = np.zeros((K, NV), np.float32)
    for i in range(0, NV, 200000): P[:, i:i+200000] = ann.predict_proba(F[:, VR[i:i+200000], VC[i:i+200000]].T).T
    return P

for seed in SEEDS:
    need = [f'{MODEL_TAG}_hind2023_seed{seed}.tif'] + [f'{MODEL_TAG}_fore2033_{sc}_seed{seed}.tif' for sc in SCEN]
    if all(os.path.exists(os.path.join(OUT, f)) for f in need): print(f'seed {seed}: all outputs already exist -> skipped (resume)'); continue
    print(f'\n===== {MODEL_TAG} | seed {seed} =====')
    # ---- Stage V: out-of-time validation ----
    ann = fit_ann('2003', '2013', seed)
    hind = scatter(ca_allocate(proba_valid(ann, feats('2013')), LULC['2013'], P_T1_train))
    save_map(hind, f'{MODEL_TAG}_hind2023_seed{seed}.tif')
    r = evaluate_map(LULC['2023'], hind, LULC['2013'], TEST)
    print(f'  [VALIDATION 2023, test blocks] OA {r["OA"]*100:.2f}% | Kappa {r["Kappa"]:.4f} | FoM {r["FoM"]:.4f} | change predicted {r["change_predicted"]*100:.2f}% (observed {r["change_observed"]*100:.2f}%)')
    # ---- Stage F: forecast ----
    ann = fit_ann('2013', '2023', seed); P = proba_valid(ann, feats('2023'))
    for sc, Pm in SCEN.items():
        print(f'  forecast scenario {sc}')
        fore = scatter(ca_allocate(P, LULC['2023'], Pm)); save_map(fore, f'{MODEL_TAG}_fore2033_{sc}_seed{seed}.tif')
print('\nDone. Maps written to', OUT)
